In [ ]:
# SBERT 패키지
!pip install -q sentence-transformers

In [ ]:
# KoGPT 패키지
!pip install transformers

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile
import json
import io
import glob
import torch
import shutil
import pickle
import pandas as pd
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM, TextDataset, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from transformers import pipeline
from datasets import Dataset
from tqdm import tqdm
from glob import glob

## 📌 **1. 데이터 만지기**

In [ ]:
# 정확한 디렉토리 경로
train_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/Training"
valid_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/Validation"
zip_files = glob(os.path.join(train_dir, "*.zip")) + glob(os.path.join(valid_dir, "*.zip"))

qa_pairs = []

for zip_path in zip_files:
    print(f"📦 열기: {zip_path}")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for name in zip_ref.namelist():
                if name.endswith(".json"):
                    with zip_ref.open(name) as raw:
                        with io.TextIOWrapper(raw, encoding='utf-8') as f:
                            try:
                                data = json.load(f)
                                for session in data.get("sessionInfo", []):
                                    dialog = session.get("dialog", [])
                                    for i in range(len(dialog) - 1):
                                        s1 = dialog[i].get("speaker")
                                        s2 = dialog[i+1].get("speaker")
                                        if s1 == "speaker1" and s2 == "speaker2":
                                            q = dialog[i].get("utterance", "").strip()
                                            a = dialog[i+1].get("utterance", "").strip()
                                            if q and a:
                                                qa_pairs.append((q, a))
                            except Exception as e:
                                print(f"[!] JSON 파싱 오류: {name} in {zip_path} - {e}")
    except Exception as e:
        print(f"[!] Zip 열기 실패: {zip_path} - {e}")

print(f"\n✅ 최종 수집된 QA 쌍 수: {len(qa_pairs)}")

In [ ]:
# SBERT용 데이터

df_sbert = pd.DataFrame(qa_pairs, columns=["query", "response"])
df_sbert.to_csv("sbert_data.tsv", sep="\t", index=False)

print("✅ SBERT용 TSV 저장 완료: sbert_data.tsv")

In [ ]:
# KoGPT용 데이터

with open("kogpt2_data.jsonl", "w", encoding="utf-8") as f:
    for q, a in qa_pairs:
        item = {"prompt": q, "response": a}
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

print("✅ KoGPT-2용 JSONL 저장 완료: kogpt2_data.jsonl")

## 🤖 **SBERT ChatBot**

In [ ]:
# 1. 데이터 불러오기

df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_data.tsv", sep='\t')
questions = df["query"].tolist()

In [ ]:
# 2. SBERT 모델 불러오기

model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")

In [ ]:
# 3. 임베딩 처리

q_embeddings = model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True,
    device="cuda"  # GPU 직접 지정
)

# 4. 저장
torch.save(q_embeddings, "sbert_question_embeddings.pt")
print("✅ 임베딩 저장 완료")

In [ ]:
# 위 과정 완료 시 여기부터.

df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_data.tsv", sep='\t')
questions = df["query"].tolist()
answers = df["response"].tolist()

model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")

q_embeddings = torch.load("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_question_embeddings.pt")

In [ ]:
# 5. GPU 설정

device = "cuda" if torch.cuda.is_available() else "cpu"
q_embeddings = q_embeddings.to(device)
model.to(device)

In [ ]:
# 6. 챗봇과 대화

chat_history = []

print("🤖 질문을 입력하세요. (종료하려면 '끝' 입력)")

while True:
    user_input = input("🙋‍♀️ ME : ").strip()

    if user_input == "끝":
        print("👋 대화를 종료합니다. 좋은 하루 보내세요!")
        break

    # 이전 대화 맥락 포함 입력 구성 (최대 3턴까지)
    full_input = ""
    for q, a in chat_history[-3:]:
        full_input += f"Q: {q}\nA: {a}\n"
    full_input += f"Q: {user_input}"

    # 임베딩 입력
    input_emb = model.encode(full_input, convert_to_tensor=True, device=device)

    # 코사인 유사도 계산
    sim = F.cosine_similarity(input_emb, q_embeddings)

    # 가장 유사한 질문 찾기
    best_idx = torch.argmax(sim).item()
    response = answers[best_idx]

    # 응답 출력 및 기록
    print("🤖 챗봇:", response)
    chat_history.append((user_input, response))

##🤖 **KoGPT ChatBot**

In [ ]:
# 1. 데이터 불러오기

def unzip_all_in_folder(zip_dir, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    zip_files = [os.path.join(zip_dir, f) for f in os.listdir(zip_dir) if f.endswith('.zip')]
    for zf in zip_files:
        with zipfile.ZipFile(zf, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    print(f"✅ 압축 해제 완료: {extract_to}")

# 경로 설정
train_zip_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/Training"
val_zip_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/Validation"
train_extract_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/train_json"
val_extract_dir = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/val_json"

unzip_all_in_folder(train_zip_dir, train_extract_dir)
unzip_all_in_folder(val_zip_dir, val_extract_dir)

In [ ]:
# 2. 다중턴 대화 추출 및 저장

def collect_multiturn_dialogs(json_dir):
    dialogs = []
    json_files = [os.path.join(json_dir, f) for f in os.listdir(json_dir) if f.endswith(".json")]

    for file in tqdm(json_files):
        with open(file, encoding="utf-8") as f:
            data = json.load(f)
            for sess in data.get("sessionInfo", []):
                d = ""
                for utt in sess.get("dialog", []):
                    spk = utt.get("speaker")
                    txt = utt.get("utterance", "").strip()
                    if not txt: continue
                    if spk == "speaker1":
                        d += f"사용자: {txt}\n"
                    elif spk == "speaker2":
                        d += f"챗봇: {txt}\n"
                if d.strip():
                    dialogs.append(d.strip())
    return dialogs

train_dialogs = collect_multiturn_dialogs(train_extract_dir)
val_dialogs = collect_multiturn_dialogs(val_extract_dir)

# 저장
with open("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/train_multiturn.txt", "w", encoding="utf-8") as f:
    for d in train_dialogs:
        f.write(d + "\n\n")

with open("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/val_multiturn.txt", "w", encoding="utf-8") as f:
    for d in val_dialogs:
        f.write(d + "\n\n")

In [ ]:
# 3. 학습 준비

model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.resize_token_embeddings(len(tokenizer))

def get_dataset(file_path):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=128,
        overwrite_cache=True
    )

In [ ]:
# 4. 파인튜닝

train_dataset = get_dataset("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/train_multiturn.txt")
val_dataset = get_dataset("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/val_multiturn.txt")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/kogpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    logging_strategy="steps",
    disable_tqdm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
trainer.save_model("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/kogpt2-finetuned")
tokenizer.save_pretrained("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/kogpt2-finetuned")

In [ ]:
## 다음부터 실행 시 여기부터

model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/kogpt2-finetuned")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/kogpt2-finetuned")
model.eval()
model.to("cuda")

In [ ]:
# 5. 멀티턴 대화 생성

def chat(prompt, max_length=60):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1] + 30,
        do_sample=True,
        top_p=0.92,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    gen = result[len(prompt):].strip()
    for sw in ["사용자:", "챗봇:"]:
        if sw in gen:
            gen = gen.split(sw)[0].strip()
    return gen.split("\n")[0].strip()


In [ ]:
# 6. 챗봇과 대화

print("챗봇 모드 시작! (종료하려면 '종료' 입력)")

history = ""
while True:
    user = input("나: ")
    if user.strip().lower() in ["종료", "exit", "quit"]:
        print("대화를 종료합니다.")
        break
    history += f"사용자: {user}\n챗봇:"
    reply = chat(history)
    print("챗봇:", reply)
    history += f"{reply}\n"

## SBERT 평가

In [ ]:
pip install bert-score

In [ ]:
# 평가 데이터 만들기

original_df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_data.tsv", sep="\t")
original_df = original_df[["query", "response"]]

# 랜덤 샘플 500개 추출
eval_df = original_df.sample(n=500, random_state=42).reset_index(drop=True)

# 평가셋 저장
eval_path = "/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/eval_set.csv"
eval_df.to_csv(eval_path, index=False)

eval_df.head()

In [ ]:
# 임베딩 불러오기
q_embeddings = torch.load("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_question_embeddings.pt").to("cuda")

# 질문-응답 리스트
df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/sbert_data.tsv", sep='\t')
questions = df["query"].tolist()
answers   = df["response"].tolist()

# 평가 함수
def sbert_bot(user_question: str) -> str:
    input_emb = model.encode(user_question, convert_to_tensor=True, device="cuda")
    sim = F.cosine_similarity(input_emb, q_embeddings)
    best_idx = torch.argmax(sim).item()
    return answers[best_idx]

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score

model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")

# 평가셋 로딩
eval_df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/eval_set.csv")
eval_questions = eval_df["query"].tolist()
eval_answers   = eval_df["response"].tolist()

# 예측 수집
sbert_preds = [sbert_bot(q) for q in eval_questions]

# BLEU 계산
def compute_bleu(pred, ref):
    return sentence_bleu([ref.split()], pred.split(), weights=(0.5, 0.5))

bleu_sbert = sum(compute_bleu(p, r) for p, r in zip(sbert_preds, eval_answers)) / len(eval_answers)

print(f"🔹 SBERT 평균 BLEU: {bleu_sbert:.4f}")

# BERTScore 계산
P_s, R_s, F1_s = score(sbert_preds, eval_answers, lang="ko", verbose=True)

print(f"🟢 SBERT BERTScore F1: {F1_s.mean().item():.4f}")